In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high_small_q5/checkpoints/post_cell_37_try_0.pickle

In [ ]:
%%cudf.pandas.profile
### cell 38 ###

### cell 38 optimized for cudf ###
def grab_subset_of_data_50(original_df, question_of_interest):
    # 1) GPU‐accelerated column filtering (no Python loop)
    subset_df = original_df.filter(like=question_of_interest)

    # 2) GPU‐vectorized string ops to extract the suffix after the last “-” and strip whitespace
    #    .to_series() turns the Index into a Series so we can use .str
    new_names = (
        subset_df.columns.to_series()
            .str.split('-')
            .str.get(-1)
            .str.lstrip()
            .to_list()
    )
    subset_df.columns = new_names

    # 3) Drop rows where *all* of these new columns are null (GPU)
    #    (omitting subset= since we only have these cols in the df)
    return subset_df.dropna(how='all')

question_of_interest_cell50 = \
    'Which of the following natural language processing (NLP) methods do you use on a regular basis?'
nlp_df_2022 = grab_subset_of_data_50(responses_df_2022_cell10, question_of_interest_cell50)
nlp_df_2022.info()